# IoT Device Classification and Location Prediction

## Manifest-Based Multi-Output CNN Using RF Spectrograms

This notebook mirrors the maintained `src/` pipeline. It trains a dual-head CNN that predicts:

- **Device type**: `dooralarm`, `lora`, `microphone`, `mbus`, `sigfox`, `miwi`
- **Location/scenario**: `anotherroom`, `sameroom`, `upstairs`

Latest recorded file-level test result from `results/test/metrics.json`:

- **Device detection**: `100.00%` (`18/18` held-out recordings)
- **Location detection**: `72.22%` (`13/18` held-out recordings)

The production scripts remain the source of truth. This notebook imports those modules instead of maintaining a separate copy of the training code.

## Setup

Expected project layout:

```text
iot-device-classification/
├── data/
│   ├── recordings_manifest.csv
│   └── recordings/<class>/<scenario>_captureXX.npy
├── src/
├── models/
└── results/
```

Install dependencies in a Python 3.10 environment:

```bash
python -m pip install -r requirements.txt
```

The recommended training run for a 48 GB system uses `MAX_WINDOWS_PER_CLASS = 10000`. Increase only after confirming memory headroom.

In [ ]:
from __future__ import annotations

import csv
import json
import os
import sys
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import tensorflow as tf
from sklearn.metrics import classification_report, confusion_matrix
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau

# Resolve project root whether the notebook is launched from the repo root or notebooks/.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from aggregation import AGGREGATION_MODES
from config import BATCH_SIZE, CLASSES, EPOCHS, LOCATIONS, NUM_CLASSES, NUM_LOCATIONS, SEED, WINDOW_SIZE
from dataset import build_dataset, load_manifest
from evaluation import save_confusion_matrix_plot, save_training_curves, write_classification_report
from infer import predict_signal
from model import StopGradientLayer, build_cnn  # StopGradientLayer registers custom layer serialization.
from preprocessing import class_distribution, segment_signal, windows_to_model_input
from test import run_test

np.random.seed(SEED)
tf.random.set_seed(SEED)

MANIFEST_PATH = Path("data/recordings_manifest.csv")
OUTPUT_DIR = Path("models/notebook_manifest_avgmax_10000")
RESULTS_DIR = Path("results/notebook_manifest_avgmax_10000")
POOLING = "avgmax"
MAX_WINDOWS_PER_CLASS = 10000
LEARNING_RATE = 3e-4
AGGREGATION = "mean"
TOP_FRACTION = 0.25

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Project root: {PROJECT_ROOT}")
print(f"Manifest: {MANIFEST_PATH}")
print(f"Classes: {CLASSES}")
print(f"Locations: {LOCATIONS}")
print(f"Max windows per class: {MAX_WINDOWS_PER_CLASS}")

## Manifest Validation

The manifest assigns whole recordings to `train`, `validation`, and `test` before segmentation. This prevents windows from the same capture appearing in multiple splits.

In [ ]:
recordings = load_manifest(MANIFEST_PATH, classes=CLASSES)

split_counts = Counter(r.split or "unspecified" for r in recordings)
scenario_counts = Counter(r.scenario for r in recordings)
class_counts = Counter(r.class_name for r in recordings)
class_by_split = defaultdict(Counter)
scenario_by_split = defaultdict(Counter)
paths = Counter(str(r.path.resolve()) for r in recordings)

missing_files = [str(r.path) for r in recordings if not r.path.exists()]
duplicate_paths = [path for path, count in paths.items() if count > 1]

for r in recordings:
    class_by_split[r.split][r.class_name] += 1
    scenario_by_split[r.split][r.scenario] += 1

print(f"Total recordings: {len(recordings)}")
print(f"By split: {dict(sorted(split_counts.items()))}")
print(f"By scenario: {dict(sorted(scenario_counts.items()))}")
print(f"By class: {dict((c, class_counts[c]) for c in CLASSES)}")
print(f"Missing files: {len(missing_files)}")
print(f"Duplicate paths: {len(duplicate_paths)}")

for split in ("train", "validation", "test"):
    print(f"
{split.upper()}")
    print("  classes:", dict((c, class_by_split[split][c]) for c in CLASSES))
    print("  scenarios:", dict((s, scenario_by_split[split][s]) for s in LOCATIONS))

## Dataset Construction

Training, validation, and held-out test windows are built through `src.dataset.build_dataset`. The current pipeline:

1. Loads raw `.npy` signal recordings.
2. Segments each signal into non-overlapping `4096`-sample windows for training/evaluation.
3. Normalizes every window independently.
4. Converts windows to normalized `257 x 61 x 1` log-compressed spectrograms.
5. Reservoir-samples up to `MAX_WINDOWS_PER_CLASS` windows per class and split.
6. Balances classes by truncating each class to the same window count.

In [ ]:
print("Building training set...")
X_train, y_dev_train, y_loc_train = build_dataset(
    manifest_path=MANIFEST_PATH,
    split="train",
    classes=CLASSES,
    locations=LOCATIONS,
    balance=True,
    seed=SEED,
    max_windows_per_class=MAX_WINDOWS_PER_CLASS,
)

print("Building validation set...")
X_val, y_dev_val, y_loc_val = build_dataset(
    manifest_path=MANIFEST_PATH,
    split="validation",
    classes=CLASSES,
    locations=LOCATIONS,
    balance=True,
    seed=SEED,
    max_windows_per_class=MAX_WINDOWS_PER_CLASS,
)

print("Building test set...")
X_test, y_dev_test, y_loc_test = build_dataset(
    manifest_path=MANIFEST_PATH,
    split="test",
    classes=CLASSES,
    locations=LOCATIONS,
    balance=True,
    seed=SEED,
    allow_missing=True,
    max_windows_per_class=MAX_WINDOWS_PER_CLASS,
)

print(f"X_train: {X_train.shape}, device distribution: {class_distribution(y_dev_train)}")
print(f"X_val:   {X_val.shape}, device distribution: {class_distribution(y_dev_val)}")
print(f"X_test:  {X_test.shape}, device distribution: {class_distribution(y_dev_test)}")
print(f"Location test distribution: {class_distribution(y_loc_test)}")

## Spectrogram Visualization

The plot below uses the same preprocessing path supplied to the model.

In [ ]:
train_recs = load_manifest(MANIFEST_PATH, classes=CLASSES, split="train")

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
for ax, class_name in zip(axes.ravel(), CLASSES):
    rec = next(r for r in train_recs if r.class_name == class_name)
    raw = np.load(rec.path)
    window = segment_signal(raw, window_size=WINDOW_SIZE)[0:1]
    spec = windows_to_model_input(window)[0, :, :, 0]
    im = ax.imshow(spec, aspect="auto", origin="lower", cmap="viridis")
    ax.set_title(f"{class_name} / {rec.scenario}")
    ax.set_xlabel("time bin")
    ax.set_ylabel("frequency bin")
fig.colorbar(im, ax=axes.ravel().tolist(), shrink=0.75)
plt.tight_layout()
plt.show()

## Multi-Output CNN Architecture

The model is built by `src.model.build_cnn`:

- Four convolutional feature blocks with batch normalization and ReLU.
- `avgmax` pooling: concatenated global average and global max pooling.
- Device head: `Dense(256)` + dropout + six-way softmax.
- Location head: `StopGradientLayer` + private dense branch + three-way softmax.
- Loss weights: `device=1.0`, `location=0.5`.

`StopGradientLayer` prevents the location objective from updating the shared backbone, protecting device features from location-gradient interference.

In [ ]:
model = build_cnn(
    input_shape=X_train.shape[1:],
    num_classes=NUM_CLASSES,
    num_locations=NUM_LOCATIONS,
    learning_rate=LEARNING_RATE,
    pooling=POOLING,
)
model.summary()

## Model Training

The callbacks match `src/train.py`: reduce learning rate on `val_device_loss`, checkpoint/early-stop on `val_device_accuracy`, and restore the best device-accuracy weights.

In [ ]:
y_dev_train_cat = tf.keras.utils.to_categorical(y_dev_train, NUM_CLASSES)
y_dev_val_cat = tf.keras.utils.to_categorical(y_dev_val, NUM_CLASSES)
y_dev_test_cat = tf.keras.utils.to_categorical(y_dev_test, NUM_CLASSES)

y_loc_train_cat = tf.keras.utils.to_categorical(y_loc_train, NUM_LOCATIONS)
y_loc_val_cat = tf.keras.utils.to_categorical(y_loc_val, NUM_LOCATIONS)
y_loc_test_cat = tf.keras.utils.to_categorical(y_loc_test, NUM_LOCATIONS)

best_model_path = OUTPUT_DIR / "best_iot_classifier.h5"
final_model_path = OUTPUT_DIR / "final_iot_classifier.h5"

callbacks = [
    ReduceLROnPlateau(
        monitor="val_device_loss",
        factor=0.5,
        patience=3,
        min_lr=1e-6,
        mode="min",
        verbose=1,
    ),
    EarlyStopping(
        monitor="val_device_accuracy",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    ModelCheckpoint(
        str(best_model_path),
        monitor="val_device_accuracy",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
]

history = model.fit(
    X_train,
    {"device": y_dev_train_cat, "location": y_loc_train_cat},
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    validation_data=(X_val, {"device": y_dev_val_cat, "location": y_loc_val_cat}),
    callbacks=callbacks,
    verbose=1,
)

model.save(str(final_model_path))
print(f"Saved best model: {best_model_path}")
print(f"Saved final model: {final_model_path}")

## Window-Level Evaluation

Window-level metrics are useful diagnostics. They are not the same as final file-level accuracy because thousands of windows from one recording are correlated.

In [ ]:
test_metrics = model.evaluate(
    X_test,
    {"device": y_dev_test_cat, "location": y_loc_test_cat},
    verbose=0,
    return_dict=True,
)
print("Window-level test metrics:")
for name, value in test_metrics.items():
    print(f"  {name}: {value:.4f}")

device_probs, location_probs = model.predict(X_test, verbose=0)
device_pred = np.argmax(device_probs, axis=1)
location_pred = np.argmax(location_probs, axis=1)

print("
DEVICE CLASSIFICATION REPORT")
print(classification_report(y_dev_test, device_pred, labels=list(range(NUM_CLASSES)), target_names=CLASSES, zero_division=0))

print("LOCATION CLASSIFICATION REPORT")
print(classification_report(y_loc_test, location_pred, labels=list(range(NUM_LOCATIONS)), target_names=LOCATIONS, zero_division=0))

## Confusion Matrices and Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 7))
cm_device = confusion_matrix(y_dev_test, device_pred, labels=list(range(NUM_CLASSES)))
cm_location = confusion_matrix(y_loc_test, location_pred, labels=list(range(NUM_LOCATIONS)))

sns.heatmap(cm_device, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES, ax=axes[0])
axes[0].set_title("Device Confusion Matrix")
axes[0].set_xlabel("Predicted")
axes[0].set_ylabel("True")

sns.heatmap(cm_location, annot=True, fmt="d", cmap="Greens", xticklabels=LOCATIONS, yticklabels=LOCATIONS, ax=axes[1])
axes[1].set_title("Location Confusion Matrix")
axes[1].set_xlabel("Predicted")
axes[1].set_ylabel("True")

plt.tight_layout()
plt.show()

save_confusion_matrix_plot(y_dev_test, device_pred, CLASSES, OUTPUT_DIR / "device_confusion_matrix.png")
save_confusion_matrix_plot(y_loc_test, location_pred, LOCATIONS, OUTPUT_DIR / "location_confusion_matrix.png")
write_classification_report(y_dev_test, device_pred, CLASSES, OUTPUT_DIR / "device_classification_report.txt")
write_classification_report(y_loc_test, location_pred, LOCATIONS, OUTPUT_DIR / "location_classification_report.txt")

In [ ]:
save_training_curves(history, OUTPUT_DIR / "training_curves.png")

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes[0, 0].plot(history.history["device_accuracy"], label="Train")
axes[0, 0].plot(history.history["val_device_accuracy"], label="Validation")
axes[0, 0].set_title("Device Accuracy")
axes[0, 0].legend()

axes[0, 1].plot(history.history["device_loss"], label="Train")
axes[0, 1].plot(history.history["val_device_loss"], label="Validation")
axes[0, 1].set_title("Device Loss")
axes[0, 1].legend()

axes[1, 0].plot(history.history["location_accuracy"], label="Train")
axes[1, 0].plot(history.history["val_location_accuracy"], label="Validation")
axes[1, 0].set_title("Location Accuracy")
axes[1, 0].legend()

axes[1, 1].plot(history.history["location_loss"], label="Train")
axes[1, 1].plot(history.history["val_location_loss"], label="Validation")
axes[1, 1].set_title("Location Loss")
axes[1, 1].legend()

plt.tight_layout()
plt.show()

## Whole-File Inference

File-level inference uses overlapping `4096`-sample windows with step `1024`, then aggregates all per-window probabilities into one prediction per recording. The default aggregation is `mean`; additional supported modes are available in `src.aggregation`.

In [ ]:
print(f"Supported aggregation modes: {AGGREGATION_MODES}")

test_recs = load_manifest(MANIFEST_PATH, classes=CLASSES, split="test")
example = test_recs[0]
signal = np.load(example.path)
result = predict_signal(
    model,
    signal,
    CLASSES,
    locations=LOCATIONS,
    window_size=WINDOW_SIZE,
    step=1024,
    aggregation=AGGREGATION,
    top_fraction=TOP_FRACTION,
)

print(f"File: {example.path}")
print(f"True device/location: {example.class_name} / {example.scenario}")
print(json.dumps(result, indent=2))

## File-Level Test Evaluation

This is the headline metric for deployment-style performance because each held-out recording contributes one prediction.

In [ ]:
file_metrics = run_test(
    model_path=best_model_path,
    metadata_path=OUTPUT_DIR / "metadata.json",  # falls back to default classes if metadata is not present yet
    data_dir=None,
    output_dir=RESULTS_DIR,
    window_size=WINDOW_SIZE,
    step=1024,
    manifest_path=MANIFEST_PATH,
    manifest_split="test",
    aggregation=AGGREGATION,
    top_fraction=TOP_FRACTION,
)

print(json.dumps(file_metrics, indent=2))

## Save Metadata

The metadata records the experiment configuration and window-level metrics for later `test.py` or `predict.py` runs.

In [ ]:
metadata = {
    "classes": CLASSES,
    "locations": LOCATIONS,
    "split_strategy": "recording_manifest",
    "source": {
        "manifest": str(MANIFEST_PATH),
        "recordings_by_split": dict(sorted(split_counts.items())),
        "recordings_by_scenario": dict(sorted(scenario_counts.items())),
    },
    "test_evaluation_role": "held-out-recordings-from-manifest",
    "test_evaluation_unit": "non_overlapping_window",
    "test_samples": int(len(y_dev_test)),
    "test_metrics": {k: float(v) for k, v in test_metrics.items()},
    "input_shape": list(X_train.shape[1:]),
    "balanced": True,
    "max_windows_per_class": MAX_WINDOWS_PER_CLASS,
    "pooling": POOLING,
}

(OUTPUT_DIR / "metadata.json").write_text(json.dumps(metadata, indent=2), encoding="utf-8")
print(json.dumps(metadata, indent=2))